# NECOFS GOM6 Wave Model vs NERACOOS Buoy Observations

Compares the GOM6 WAVE timeseries Icechunk store (single `time` dimension) against
the five active NERACOOS directional wave buoys in the Gulf of Maine.

- **Past 2 days**: obs (black) and model (blue dashed) overlaid  
- **Forecast window**: model only — shows the current wave forecast at each buoy location  
- **Final cell**: latest model significant wave height map

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import holoviews as hv
import icechunk
import xarray as xr
import xugrid as xu
import hvplot.xugrid
import hvplot.pandas
import hvplot.xarray
import panel as pn
from scipy.spatial import cKDTree
from dotenv import load_dotenv

pn.extension()
hv.extension('bokeh')
_ = load_dotenv(os.path.expanduser('~/dotenv/gom3_forecast.env'))

## Open GOM6 Wave Timeseries Store

Uses `gom6_wave_timeseries` — a single `time` dimension (no FMRC structure needed).

In [2]:
BUCKET    = 'neracoos-necofs-forecast'
REGION    = 'us-east-1'
IC_PREFIX = 'WAVE/icechunk/gom6_wave_timeseries'

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{BUCKET}/',
        store=icechunk.s3_store(region=REGION),
    ),
)
storage     = icechunk.s3_storage(bucket=BUCKET, prefix=IC_PREFIX, region=REGION)
credentials = icechunk.containers_credentials({f's3://{BUCKET}/': icechunk.s3_credentials()})
repo        = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
session     = repo.readonly_session('main')
ds = xr.open_zarr(session.store, consolidated=False, chunks={})
ds

<xarray.Dataset> Size: 2GB
Dimensions:            (three: 3, nele: 371290, node: 207081, time: 240,
                        siglev: 46, siglay: 45)
Coordinates:
    latc               (nele) float32 1MB dask.array<chunksize=(34514,), meta=np.ndarray>
    lonc               (nele) float32 1MB dask.array<chunksize=(34514,), meta=np.ndarray>
    lat                (node) float32 828kB dask.array<chunksize=(34514,), meta=np.ndarray>
    lon                (node) float32 828kB dask.array<chunksize=(34514,), meta=np.ndarray>
    x                  (node) float32 828kB dask.array<chunksize=(34514,), meta=np.ndarray>
    y                  (node) float32 828kB dask.array<chunksize=(34514,), meta=np.ndarray>
  * time               (time) datetime64[ns] 2kB 2026-05-20 ... 2026-05-29T23...
    siglev             (siglev, node) float32 38MB dask.array<chunksize=(46, 34514), meta=np.ndarray>
    siglay             (siglay, node) float32 37MB dask.array<chunksize=(45, 34514), meta=np.ndarray>
Dimensions without coordinates: three, nele, node
Data variables: (12/17)
    aw0                (three, nele) float32 4MB dask.array<chunksize=(3, 34514), meta=np.ndarray>
    awx                (three, nele) float32 4MB dask.array<chunksize=(3, 34514), meta=np.ndarray>
    h                  (node) float32 828kB dask.array<chunksize=(34514,), meta=np.ndarray>
    hs                 (time, node) float32 199MB dask.array<chunksize=(24, 34514), meta=np.ndarray>
    awy                (three, nele) float32 4MB dask.array<chunksize=(3, 34514), meta=np.ndarray>
    mesh_topology      int32 4B ...
    ...                 ...
    yc                 (nele) float32 1MB dask.array<chunksize=(34514,), meta=np.ndarray>
    wdir               (time, node) float32 199MB dask.array<chunksize=(24, 34514), meta=np.ndarray>
    vwind_speed        (time, nele) float32 356MB dask.array<chunksize=(24, 34514), meta=np.ndarray>
    tpeak              (time, node) float32 199MB dask.array<chunksize=(24, 34514), meta=np.ndarray>
    zeta               (time, node) float32 199MB dask.array<chunksize=(24, 34514), meta=np.ndarray>
    xc                 (nele) float32 1MB dask.array<chunksize=(34514,), meta=np.ndarray>
Attributes: (12/15)
    title:                       FVCOM GOM NECOFS HINDCAST UPDATE
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.1
    references:                  http://fvcom.smast.umassd.edu, http://codfis...
    Conventions:                 CF-1.11, UGRID-1.0
    CoordinateSystem:            Cartesian
    ...                          ...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  FVCOM periodic surface precip forcing:\nFILE...
    NCO:                         netCDF Operators version 5.3.9 (Homepage = h...
    history:                     Thu May 21 15:00:58 2026: /data/rsignell/min...

## Active NERACOOS Wave Buoys

Five buoys in the Gulf of Maine with data current through today (verified via ERDDAP).
Positions are updated below from the actual obs data.

In [3]:
buoys = {
    'A01': {'dataset_id': 'A01_waves', 'label': 'A01 — Mass Bay',       'lat': 42.523, 'lon': -70.566},
    'B01': {'dataset_id': 'B01_waves', 'label': 'B01 — W Maine Shelf',  'lat': 43.181, 'lon': -70.428},
    'E01': {'dataset_id': 'E01_waves', 'label': 'E01 — C Maine Shelf',  'lat': 43.715, 'lon': -69.355},
    'F01': {'dataset_id': 'F01_waves', 'label': 'F01 — Penobscot Bay',  'lat': 44.056, 'lon': -68.998},
    'I01': {'dataset_id': 'I01_waves', 'label': 'I01 — E Maine Shelf',  'lat': 44.106, 'lon': -68.109},
}

## Fetch Last 5 Days of Observations from NERACOOS ERDDAP

In [4]:
ERDDAP_BASE  = 'https://data.neracoos.org/erddap/tabledap'
LOOKBACK     = pd.Timedelta('5D')
t_plot_start = (pd.Timestamp.utcnow() - LOOKBACK).replace(tzinfo=None)
t_start      = t_plot_start.strftime('%Y-%m-%dT%H:%M:%SZ')

obs_data = {}
for name, info in buoys.items():
    url = (
        f"{ERDDAP_BASE}/{info['dataset_id']}.csv"
        f"?time,latitude,longitude,significant_wave_height,dominant_wave_period"
        f"&time>={t_start}"
    )
    df = pd.read_csv(url, skiprows=[1], parse_dates=['time'], index_col='time')
    df = df.apply(pd.to_numeric, errors='coerce')
    # Strip tz info (ERDDAP returns UTC; keep as tz-naive to match model)
    if df.index.tz is not None:
        df.index = df.index.tz_convert('UTC').tz_localize(None)
    # Update position from actual data
    buoys[name]['lat'] = float(df['latitude'].median())
    buoys[name]['lon'] = float(df['longitude'].median())
    obs_data[name] = df[['significant_wave_height', 'dominant_wave_period']]
    print(f"{name}: {len(df):4d} records  lat={buoys[name]['lat']:.3f}  lon={buoys[name]['lon']:.3f}")

A01:  121 records  lat=42.523  lon=-70.566
B01:  121 records  lat=43.179  lon=-70.427
E01:  120 records  lat=43.715  lon=-69.358
F01:  119 records  lat=44.056  lon=-68.998
I01:  120 records  lat=44.102  lon=-68.113


## Find Nearest Model Nodes

Builds a KD-tree over the GOM6 unstructured grid node coordinates and queries
the nearest node index for each buoy.

In [5]:
model_lon = ds['lon'].compute().values
model_lat = ds['lat'].compute().values
tree = cKDTree(np.column_stack([model_lon, model_lat]))

for name, info in buoys.items():
    dist, idx = tree.query([info['lon'], info['lat']])
    buoys[name]['node_idx'] = int(idx)
    print(f"{name}: node {idx:7d}  dist={dist:.4f}deg  "
          f"model ({model_lat[idx]:.3f}N, {model_lon[idx]:.3f}E)")

A01: node   72429  dist=0.0085deg  model (42.526N, -70.558E)
B01: node   60282  dist=0.0073deg  model (43.179N, -70.434E)
E01: node   55341  dist=0.0089deg  model (43.715N, -69.367E)
F01: node   77512  dist=0.0015deg  model (44.054N, -68.999E)
I01: node   47274  dist=0.0123deg  model (44.091N, -68.109E)


## Extract Model `hs` Time Series at Buoy Nodes

Selects model data from 5 days ago through the end of the forecast at each nearest node.
Times are tz-naive UTC (matching the obs index).

In [6]:
model_hs = {}
for name, info in buoys.items():
    ts = ds['hs'].isel(node=info['node_idx']).sel(time=slice(t_plot_start, None)).compute().to_series()
    ts.name = 'GOM6 model'
    model_hs[name] = ts  # tz-naive UTC

model_hs['A01'].head()

time
2026-05-21 02:00:00    0.371704
2026-05-21 03:00:00    0.432983
2026-05-21 04:00:00    0.557129
2026-05-21 05:00:00    0.554565
2026-05-21 06:00:00    0.457336
Name: GOM6 model, dtype: float32

## Model vs Observations — Significant Wave Height

Black = buoy obs (last 2 days), blue dashed = GOM6 model (hindcast + forecast).
The red dotted line marks "now"; the model extends beyond it to show the current forecast.

In [7]:
# tz-naive UTC 'now' to match both model and obs time axes
t_now = pd.Timestamp.utcnow().replace(tzinfo=None)

plots = []
for name, info in buoys.items():
    obs_hs = obs_data[name]['significant_wave_height']

    p_obs = obs_hs.hvplot(
        label='obs', color='black', line_width=2,
        width=750, height=280,
    )
    p_mod = model_hs[name].hvplot(
        label='GOM6 model', color='steelblue', line_width=2, line_dash='dashed',
    )
    now_line = hv.VLine(t_now).opts(
        color='firebrick', line_dash='dotted', line_width=1.5,
    )

    plot = (p_obs * p_mod * now_line).opts(
        title=f'{info["label"]} — Hs (m)',
        xlabel='Time (UTC)', ylabel='Hs (m)',
        legend_position='top_left',
        show_grid=True,
    )
    plots.append(plot)

pn.Column(*plots)

Column
    [0] HoloViews(Overlay, height=280, sizing_mode='fixed', width=750)
    [1] HoloViews(Overlay, height=280, sizing_mode='fixed', width=750)
    [2] HoloViews(Overlay, height=280, sizing_mode='fixed', width=750)
    [3] HoloViews(Overlay, height=280, sizing_mode='fixed', width=750)
    [4] HoloViews(Overlay, height=280, sizing_mode='fixed', width=750)